
# Notebook 11 · Churn y reactivación con next best action

## Pregunta de negocio
¿Qué clientes muestran riesgo de abandono y qué acción conviene activar en cada caso?

## Qué aprenderás
- Cómo crear snapshots temporales para churn.
- Cómo combinar transacciones, campañas, soporte y feedback.
- Cómo pasar del score a una acción concreta.

## Outputs
- `nb11_churn_training_dataset.csv`
- `nb11_churn_scoring_latest.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

fact = tables["fact"].copy()
crm = tables["crm"].copy()
contactos = tables["contactos"].copy()
eventos_campana = tables["eventos_campana"].copy()
tickets = tables["tickets"].copy()
feedback = tables["feedback"].copy()
cupones = tables["cupones"].copy()

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["customer_id"].notna()].copy()

snapshot_dates = pd.date_range("2025-03-01", "2025-09-01", freq="MS")
rows = []
for snap in snapshot_dates:
    hist_start = snap - pd.Timedelta(days=180)
    label_end = snap + pd.Timedelta(days=90)
    for cust in sales["customer_id"].dropna().unique():
        hist = sales[(sales["customer_id"] == cust) & (sales["fecha_movimiento_dt"] < snap) & (sales["fecha_movimiento_dt"] >= hist_start)]
        if hist.empty:
            continue
        future = sales[(sales["customer_id"] == cust) & (sales["fecha_movimiento_dt"] >= snap) & (sales["fecha_movimiento_dt"] < label_end)]
        rows.append({
            "snapshot_date": snap,
            "customer_id": cust,
            "tx_180d": hist["doc_id_std"].nunique(),
            "revenue_180d": hist["importe_abs_num"].sum(),
            "margin_180d": hist["margen_bruto_num"].sum(),
            "days_since_last_purchase": (snap - hist["fecha_movimiento_dt"].max()).days,
            "returns_180d": hist["es_devolucion"].sum(),
            "online_share_180d": (hist["canal_movimiento"] == "ONLINE").mean(),
            "discount_mean_180d": hist["descuento_pct_num"].fillna(0).mean(),
            "category_count_180d": hist["categoria"].nunique(),
            "is_churn_90d": int(future["doc_id_std"].nunique() == 0),
        })

churn_base = pd.DataFrame(rows)
churn_base = churn_base[churn_base["customer_id"].isin(crm["customer_id"])].copy()
churn_base.head()


,snapshot_date,customer_id,tx_180d,revenue_180d,margin_180d,days_since_last_purchase,returns_180d,online_share_180d,discount_mean_180d,category_count_180d,is_churn_90d
1,2025-03-01,C0061,2,5152.56,1782.14,16,0,1.000000,1.666667,3,0
3,2025-03-01,C0025,1,235.60,158.08,51,0,0.000000,0.000000,1,0
4,2025-03-01,C0094,2,2173.03,744.62,16,0,0.857143,9.285714,3,0
5,2025-03-01,C0033,1,1056.08,189.35,148,0,1.000000,7.500000,1,0
6,2025-03-01,C0085,2,2199.57,651.03,48,0,1.000000,7.142857,3,1


In [3]:

def add_window_features(base_df, source_df, date_col, prefix, funs, window_days=180):
    out_rows = []
    for _, row in base_df[["customer_id", "snapshot_date"]].iterrows():
        subset = source_df[
            (source_df["customer_id"] == row["customer_id"])
            & (source_df[date_col] < row["snapshot_date"])
            & (source_df[date_col] >= row["snapshot_date"] - pd.Timedelta(days=window_days))
        ]
        feats = {}
        for name, func in funs.items():
            feats[f"{prefix}_{name}"] = func(subset)
        out_rows.append(feats)
    return pd.DataFrame(out_rows)

feat_contactos = add_window_features(
    churn_base, contactos, "fecha_contacto_dt", "contact",
    {
        "cnt": lambda s: len(s),
        "conv": lambda s: s["conversion_30d_flag"].sum(),
        "control_ratio": lambda s: s["grupo_control_flag"].mean() if len(s) else np.nan,
    }
)
feat_eventos = add_window_features(
    churn_base, eventos_campana, "fecha_evento_dt", "camp",
    {
        "opens": lambda s: (s["tipo_evento"] == "open").sum(),
        "clicks": lambda s: (s["tipo_evento"] == "click").sum(),
        "purchases": lambda s: (s["tipo_evento"] == "purchase").sum(),
    }
)
feat_tickets = add_window_features(
    churn_base, tickets, "fecha_apertura_dt", "srv",
    {
        "tickets": lambda s: len(s),
        "comp": lambda s: s["importe_compensacion_eur"].sum(),
        "csat": lambda s: s["csat_1_5"].mean(),
        "open_ratio": lambda s: s["estado_resolucion"].isin(["Pendiente", "Escalado"]).mean() if len(s) else np.nan,
    }
)
feat_feedback = add_window_features(
    churn_base, feedback, "fecha_feedback_dt", "voc",
    {
        "nps": lambda s: s["nps_0_10"].mean(),
        "rating": lambda s: s["rating_1_5"].mean(),
        "neg": lambda s: (s["sentimiento"] == "Negativo").sum(),
    }
)
feat_cupones = add_window_features(
    churn_base, cupones, "fecha_emision_dt", "coupon",
    {
        "issued": lambda s: len(s),
        "redeemed": lambda s: s["estado_cupon"].astype(str).str.lower().eq("canjeado").sum(),
        "avg_discount": lambda s: s["descuento_pct"].fillna(0).mean(),
    }
)

dataset = pd.concat([churn_base.reset_index(drop=True), feat_contactos, feat_eventos, feat_tickets, feat_feedback, feat_cupones], axis=1)
dataset = dataset.merge(
    crm[["customer_id", "tier_fidelizacion", "canal_preferido_declarado", "consentimiento_marketing", "estado_cliente"]],
    on="customer_id", how="left"
)
dataset.shape, dataset["is_churn_90d"].mean()


((511, 31), np.float64(0.4187866927592955))

In [4]:

X = dataset.drop(columns=["customer_id", "snapshot_date", "is_churn_90d"])
y = dataset["is_churn_90d"]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1200, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    "metric": ["ROC_AUC", "Average_Precision"],
    "value": [roc_auc_score(y_test, proba_test), average_precision_score(y_test, proba_test)]
})
metrics


,metric,value
0,ROC_AUC,0.659201
1,Average_Precision,0.570700


In [5]:

latest_snapshot = dataset["snapshot_date"].max()
scoring = dataset[dataset["snapshot_date"] == latest_snapshot].copy()
scoring["score_churn"] = model.predict_proba(scoring.drop(columns=["customer_id", "snapshot_date", "is_churn_90d"]))[:, 1]

def next_best_action(row):
    if row["score_churn"] >= 0.70 and row["margin_180d"] >= scoring["margin_180d"].quantile(0.75):
        return "Contacto premium + recomendación personalizada"
    if row["score_churn"] >= 0.70 and row.get("coupon_redeemed", 0) > 0:
        return "Cupón limitado con control de margen"
    if row["score_churn"] >= 0.55 and row.get("voc_neg", 0) > 0:
        return "Mensaje de servicio y recuperación de confianza"
    if row["score_churn"] >= 0.55:
        return "Email inspiracional con categoría favorita"
    return "Mantener en nurture de bajo coste"

action_factor = {
    "Contacto premium + recomendación personalizada": 0.22,
    "Cupón limitado con control de margen": 0.16,
    "Mensaje de servicio y recuperación de confianza": 0.14,
    "Email inspiracional con categoría favorita": 0.08,
    "Mantener en nurture de bajo coste": 0.03,
}

scoring["next_best_action"] = scoring.apply(next_best_action, axis=1)
scoring["expected_margin_uplift_eur"] = scoring["score_churn"] * scoring["margin_180d"] * scoring["next_best_action"].map(action_factor)
scoring = scoring.sort_values(["score_churn", "expected_margin_uplift_eur"], ascending=False)

scoring[["customer_id", "score_churn", "margin_180d", "voc_neg", "srv_tickets", "next_best_action", "expected_margin_uplift_eur"]].head(20)


,customer_id,score_churn,margin_180d,voc_neg,srv_tickets,next_best_action,expected_margin_uplift_eur
451,C0074,0.918970,437.89,0,0,Email inspiracional con categoría favorita,32.192639
433,C0014,0.842738,233.74,0,2,Email inspiracional con categoría favorita,15.758525
460,C0032,0.809688,146.62,1,0,Mensaje de servicio y recuperación de confianza,16.620301
457,C0016,0.760156,27.68,0,1,Email inspiracional con categoría favorita,1.683290
489,C0017,0.735412,24.38,0,1,Email inspiracional con categoría favorita,1.434347
504,C0070,0.735071,231.88,1,0,Mensaje de servicio y recuperación de confianza,23.862763
435,C0061,0.732230,64.17,0,0,Email inspiracional con categoría favorita,3.758974
472,C0051,0.730645,626.41,0,0,Email inspiracional con categoría favorita,36.614650
452,C0081,0.721649,595.31,0,0,Email inspiracional con categoría favorita,34.368368
466,C0024,0.716279,624.76,0,0,Email inspiracional con categoría favorita,35.800218



## Lectura de negocio

En marketing de precisión, el score no basta. La decisión real consiste en combinar:

- probabilidad de abandono,
- valor económico reciente,
- sensibilidad al incentivo,
- y señales de fricción de servicio o experiencia.

Por eso el output final es una lista priorizada con *next best action*.


In [6]:

path_train = mpu.export_csv(dataset, "nb11_churn_training_dataset.csv")
path_score = mpu.export_csv(scoring, "nb11_churn_scoring_latest.csv")

print("Guardado:", path_train)
print("Guardado:", path_score)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb11_churn_training_dataset.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb11_churn_scoring_latest.csv
